## 1. Create aws_stage in staging_layer
Configure access to an S3 bucket to load sales data

In [ ]:
-- USE DATABASE SALED_ANALYTICS_DB;

-- CREATE STAGE IF NOT EXISTS STAGING_LAYER.AWS_STAGE
--     URL = 's3://engineering-course-9/DataWarehouseAndSnowflake/'
--     CREDENTIALS = (
--         AWS_KEY_ID = ''
--         AWS_SECRET_KEY = ''
--     );

-- CREATE STAGE IF NOT EXISTS STAGING_LAYER.LOCAL_STAGE;

-- LIST @STAGING_LAYER.AWS_STAGE;

## 2. CREATE FILE FORMAT CSV
Set a consistent format to parse all CSVs from S3

In [ ]:
-- CREATE OR REPLACE FILE FORMAT STAGING_LAYER.CSV_FORMAT
--     TYPE = CSV
--     FIELD_DELIMITER = ','
--     SKIP_HEADER = 1;

## 3. Create Raw Table
Create a structure that matches the CSV file layout

In [ ]:
-- CREATE OR REPLACE TABLE RAW_LAYER.SALES_ORDERS_RAW (
--     ORDER_ID STRING,
--     CUSTOMER_ID STRING,
--     PRODUCT_ID STRING,
--     ORDER_DATE STRING,
--     AMOUNT STRING,
--     PROFIT STRING,
--     QUANTITY STRING,
--     CATEGORY STRING,
--     SUBCATEGORY STRING
-- );

## 4. LOAD DATA INTO ORDERS TABLE
Use the external stage and file format to ingest CSV data

In [ ]:
-- COPY INTO RAW_LAYER.SALES_ORDERS_RAW
--     FROM @STAGING_LAYER.AWS_STAGE
--     FILE_FORMAT = STAGING_LAYER.CSV_FORMAT
--     FILES = ('large_orders.csv')
--     ON_ERROR = 'ABORT_STATEMENT';

-- COPY INTO RAW_LAYER.ORDERS_RAW
--     FROM @STAGING_LAYER.AWS_STAGE
--     FILE_FORMAT = STAGING_LAYER.CSV_FORMAT
--     PATTERN = '.*orders_.*\.csv'
--     ON_ERROR = 'ABORT_STATEMENT';

-- LIST @STAGING_LAYER.AWS_STAGE;

-- SELECT * FROM RAW_LAYER.SALES_ORDERS_RAW;

## 5. Create Cleansed Table
Clean data types and apply basic formatting

In [ ]:
-- CREATE TRANSIENT TABLE CLEANSED_LAYER.SALES_ORDERS_CLEAN AS
-- SELECT
--     ORDER_ID,
--     CUSTOMER_ID,
--     PRODUCT_ID,
--     TO_DATE(ORDER_DATE, 'YYYY-MM-DD')   AS ORDER_DATE,
--     TRY_TO_DECIMAL(AMOUNT, 10, 2)       AS AMOUNT,
--     TRY_TO_DECIMAL(PROFIT, 10, 2)       AS PROFIT,
--     TRY_TO_NUMBER(QUANTITY)             AS QUANTITY,
--     INITCAP(CATEGORY)                   AS CATEGORY,
--     INITCAP(SUBCATEGORY)                AS SUBCATEGORY
-- FROM RAW_LAYER.SALES_ORDERS_RAW;

-- SELECT * FROM CLEANSED_LAYER.SALES_ORDERS_CLEAN;

## 6. Mask amount unless user has PII access
Obscure amount for users without the right role

In [ ]:
-- CREATE MASKING POLICY IF NOT EXISTS CLEANSED_LAYER.MASKED_AMOUNT
--     AS (VAL DECIMAL(10,2))
--     RETURNS DECIMAL(10,2) ->
--         CASE
--             WHEN CURRENT_ROLE() = 'FINANCE_DEPARTMENT' THEN VAL
--             ELSE -1
--         END;

-- ALTER TABLE CLEANSED_LAYER.SALES_ORDERS_CLEAN
--     MODIFY COLUMN AMOUNT
--     SET MASKING POLICY CLEANSED_LAYER.MASKED_AMOUNT;

-- SHOW MASKING POLICIES;

-- SELECT * FROM TABLE(
--     INFORMATION_SCHEMA.POLICY_REFERENCES(
--         REF_ENTITY_NAME   => 'CLEANSED_LAYER.SALES_ORDERS_CLEAN',
--         REF_ENTITY_DOMAIN => 'TABLE'
--     )
-- );

-- ALTER TABLE CLEANSED_LAYER.SALES_ORDERS_CLEAN
--     MODIFY COLUMN AMOUNT
--     UNSET MASKING POLICY;

-- DROP MASKING POLICY CLEANSED_LAYER.MASKED_AMOUNT;

### 6.1 Masking policy върху текстова колона (CATEGORY)

In [ ]:
-- CREATE MASKING POLICY CLEANSED_LAYER.MASKED_CATEGORY
--     AS (VAL STRING)
--     RETURNS STRING ->
--         CASE
--             WHEN CURRENT_ROLE() = 'ACCOUNTADMIN' THEN '*****'
--             ELSE VAL
--         END;

-- ALTER TABLE CLEANSED_LAYER.SALES_ORDERS_CLEAN
--     MODIFY COLUMN CATEGORY
--     SET MASKING POLICY CLEANSED_LAYER.MASKED_CATEGORY;

-- SELECT * FROM CLEANSED_LAYER.SALES_ORDERS_CLEAN;

-- ALTER TABLE CLEANSED_LAYER.SALES_ORDERS_CLEAN
--     MODIFY COLUMN CATEGORY
--     UNSET MASKING POLICY;

-- SELECT * FROM CLEANSED_LAYER.SALES_ORDERS_CLEAN;

## 7. Create Table for MV from base table
Материализираната view не може да чете от таблица с активни политики — затова се прави чиста базова таблица

In [ ]:
-- CREATE TRANSIENT TABLE CLEANSED_LAYER.SALES_ORDERS_MV_READY AS
-- SELECT
--     ORDER_ID,
--     CUSTOMER_ID,
--     PRODUCT_ID,
--     ORDER_DATE,
--     AMOUNT,
--     PROFIT,
--     QUANTITY,
--     CATEGORY,
--     SUBCATEGORY
-- FROM CLEANSED_LAYER.SALES_ORDERS_CLEAN;

## 8. Pre-compute total sales per category for fast BI access
Aggregate sales by category for faster dashboard performance

In [ ]:
-- CREATE MATERIALIZED VIEW BUSINESS_LAYER.SALES_ORDERS_SUMMARY_MV AS
-- SELECT
--     CATEGORY,
--     SUM(AMOUNT) AS TOTAL_SALES
-- FROM CLEANSED_LAYER.SALES_ORDERS_MV_READY
-- GROUP BY CATEGORY;

-- SELECT * FROM BUSINESS_LAYER.SALES_ORDERS_SUMMARY_MV;

## 9. Create a logical view in the Presentation Layer that generates a Profit Dashboard

In [ ]:
-- CREATE OR REPLACE VIEW PRESENTATION_LAYER.TOTAL_SUMMARY_V AS
-- SELECT
--     CATEGORY,
--     TOTAL_SALES
-- FROM BUSINESS_LAYER.SALES_ORDERS_SUMMARY_MV
-- ORDER BY TOTAL_SALES DESC;

-- SELECT * FROM PRESENTATION_LAYER.TOTAL_SUMMARY_V;

## 10. Clone a Table for Testing
Duplicate a table without using additional storage

In [ ]:
-- CREATE TEMPORARY TABLE SALES_ORDERS_CLONE
--     CLONE CLEANSED_LAYER.SALES_ORDERS_CLEAN;

-- SELECT * FROM SALES_ORDERS_CLONE;

## 11. Create File Format JSON, create table, parse and insert data

In [ ]:
-- CREATE OR REPLACE FILE FORMAT STAGING_LAYER.JSON_FORMAT
--     TYPE = JSON
--     STRIP_OUTER_ARRAY = TRUE
--     ENABLE_OCTAL = FALSE;

-- CREATE OR REPLACE TABLE RAW_LAYER.REVIEWS_RAW (
--     RAW_RECORD VARIANT
-- );

-- COPY INTO RAW_LAYER.REVIEWS_RAW(RAW_RECORD)
--     FROM @STAGING_LAYER.AWS_STAGE
--     FILES = ('large_customer_reviews.json')
--     FILE_FORMAT = STAGING_LAYER.JSON_FORMAT
--     ON_ERROR = 'CONTINUE';

-- SELECT * FROM RAW_LAYER.REVIEWS_RAW;

-- CREATE TABLE CLEANSED_LAYER.REVIEWS AS
-- SELECT
--     RAW_RECORD:review_id::INT                AS REVIEW_ID,
--     RAW_RECORD:customer_id::STRING           AS CUSTOMER_ID,
--     RAW_RECORD:review_data.rating::INT       AS RATING,
--     RAW_RECORD:review_data.comment::STRING   AS COMMENT,
--     RAW_RECORD:review_data.verified::BOOLEAN AS VERIFIED
-- FROM RAW_LAYER.REVIEWS_RAW;

-- SELECT * FROM CLEANSED_LAYER.REVIEWS;

## 12. Restrict data visibility based on user category
Only see rows from their category

In [ ]:
-- CREATE ROLE FURNITURE;

-- CREATE ROW ACCESS POLICY CLEANSED_LAYER.CATEG_FILTER
--     AS (VAL STRING)
--     RETURNS BOOLEAN ->
--         CASE
--             WHEN UPPER(TRIM(VAL)) = UPPER(CURRENT_ROLE()) THEN TRUE
--             ELSE FALSE
--         END;

-- GRANT ROLE FURNITURE TO USER DIMITAR;

-- ALTER TABLE CLEANSED_LAYER.SALES_ORDERS_CLEAN
--     ADD ROW ACCESS POLICY CLEANSED_LAYER.CATEG_FILTER
--     ON (CATEGORY);

-- SELECT * FROM CLEANSED_LAYER.SALES_ORDERS_CLEAN;

## 13. Restore or query table as it existed 10 minutes ago
Review the state of the sales table 10 minutes ago

In [ ]:
-- SELECT * FROM CLEANSED_LAYER.SALES_ORDERS_CLEAN;

-- SELECT * FROM CLEANSED_LAYER.SALES_ORDERS_CLEAN AT (OFFSET => -600);

## 14. Check how much storage your tables use
Identify which tables consume the most storage

In [ ]:
-- SELECT
--     TABLE_SCHEMA,
--     TABLE_NAME,
--     ACTIVE_BYTES / 1024 / 1024 AS SIZE_MB
-- FROM SNOWFLAKE.ACCOUNT_USAGE.TABLE_STORAGE_METRICS
-- WHERE TABLE_SCHEMA IN ('RAW_LAYER', 'CLEANSED_LAYER', 'BUSINESS_LAYER')
-- ORDER BY SIZE_MB DESC;

## 15. Stop compute if credits go above limit
Auto-suspend compute after reaching quota

In [ ]:
-- CREATE RESOURCE MONITOR TEAM_LIMIT
--     WITH CREDIT_QUOTA = 50
--     TRIGGERS ON 90 PERCENT DO SUSPEND;